<a href="https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup CELL
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzad-jatoi/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd, numpy as np, json, os

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
print("Setup done. Working dir:", os.getcwd())


Setup done. Working dir: /content/flyrank-ml-internship-starter


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Primary: Logistic Regression.** My target  "will this page's ranking get worse next month?" is binary, and logistic regression gives interpretable coefficients I can defend to a non-technical reader: each feature's sign and magnitude directly says whether it pushes a page toward "likely to decline." That matches the decision-support framing from Week 1-2 (a content lead needs to trust *why* a page was flagged, not just accept a score).

**Secondary comparison: Decision Tree (depth-limited).** Included to check whether the relationship is meaningfully non-linear mirrors the hand-rule-vs-tree comparison from notebook 02, now applied to a real predictive task instead of the starter sample. I'm not using Random Forest or Gradient Boosting here added complexity isn't earning its keep yet given how few features and how modest a signal I found in Week 4's baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-aware split, not random.** Random splitting here would leak: a page's April behavior is correlated with its March behavior, so a random split would let the model see near-duplicate information across train and test.

Instead: **train** on pages using **February 2026** features to predict whether their average position got worse by **March 2026**. **Test** on a completely separate time window: **April 2026** features predicting the outcome by **May 2026**. This means the test period is entirely after the training period, and no content-level information from the test window ever touches training matching how this would actually be deployed (score current pages, wait, see what happened).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def load_features(month_str, ref_date):
    fact_path = f"{WAREHOUSE}/fact_content_daily_performance/month={month_str}/data_0.parquet"
    df = con.sql(f"""
        WITH agg AS (
            SELECT content_hash_id,
                   AVG(gsc_impressions) as avg_impressions,
                   AVG(gsc_clicks) as avg_clicks,
                   AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) as avg_ctr,
                   AVG(gsc_avg_position) as avg_position
            FROM read_parquet('{fact_path}')
            WHERE gsc_data_available IS TRUE
            GROUP BY content_hash_id
        )
        SELECT a.*,
               c.word_count, c.search_volume, c.competition,
               DATE_DIFF('day', c.content_updated_date, DATE '{ref_date}') as days_since_update
        FROM agg a
        JOIN read_parquet('{DIM_CONTENT}') c ON a.content_hash_id = c.content_hash_id
        WHERE c.is_deleted IS NOT TRUE AND a.avg_impressions >= 10
    """).df()
    return df

def load_outcome_position(month_str):
    fact_path = f"{WAREHOUSE}/fact_content_daily_performance/month={month_str}/data_0.parquet"
    return con.sql(f"""
        SELECT content_hash_id, AVG(gsc_avg_position) as future_avg_position
        FROM read_parquet('{fact_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()

# Train: Feb features -> March outcome
train_X = load_features("2026-02", "2026-02-28")
train_outcome = load_outcome_position("2026-03")
train_df = train_X.merge(train_outcome, on="content_hash_id", how="inner")
train_df["declined"] = (train_df["future_avg_position"] > train_df["avg_position"]).astype(int)

# Test: April features -> May outcome (fully separate time window)
test_X = load_features("2026-04", "2026-04-30")
test_outcome = load_outcome_position("2026-05")
test_df = test_X.merge(test_outcome, on="content_hash_id", how="inner")
test_df["declined"] = (test_df["future_avg_position"] > test_df["avg_position"]).astype(int)

FEATURES = ["avg_impressions", "avg_clicks", "avg_ctr", "avg_position", "word_count", "search_volume", "days_since_update"]

print(f"Train rows: {len(train_df)}  |  decline rate: {train_df['declined'].mean():.3f}")
print(f"Test rows:  {len(test_df)}  |  decline rate: {test_df['declined'].mean():.3f}")
train_df[FEATURES + ["declined"]].head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 62935  |  decline rate: 0.704
Test rows:  81429  |  decline rate: 0.804


,avg_impressions,avg_clicks,avg_ctr,avg_position,word_count,search_volume,days_since_update,declined
0,36.142857,0.107143,0.003350,29.609070,2828,320,-87,1
1,57.071429,0.250000,0.004062,17.806923,2689,70,-76,1
2,30.750000,0.000000,0.000000,7.852945,2928,70,-88,0
3,10.928571,0.000000,0.000000,7.965842,2288,70,-88,0
4,102.428571,0.107143,0.001044,17.993308,2934,50,-76,0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")
X_train = imputer.fit_transform(train_df[FEATURES])
y_train = train_df["declined"].values
X_test = imputer.transform(test_df[FEATURES])
y_test = test_df["declined"].values

logreg = LogisticRegression(max_iter=1000, class_weight="balanced").fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:, 1]
tree_scores = tree.predict_proba(X_test)[:, 1]

# Re-run the SAME Week-4 baseline rule on this test set
baseline_scores = (test_df["avg_impressions"] *
                    (test_df["days_since_update"] / 365.0) /
                    (test_df["avg_ctr"] + 0.005)).values

def precision_at_k(scores, labels, k):
    order = np.argsort(-scores)[:k]
    return labels[order].mean()

results = []
for k in [10, 20, 50]:
    results.append({
        "K": k,
        "baseline_rule": round(precision_at_k(baseline_scores, y_test, k), 3),
        "logistic_regression": round(precision_at_k(logreg_scores, y_test, k), 3),
        "decision_tree": round(precision_at_k(tree_scores, y_test, k), 3),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


 K  baseline_rule  logistic_regression  decision_tree
10           0.60                 0.90           1.00
20           0.65                 0.95           1.00
50           0.66                 0.92           0.96


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.inspection import permutation_importance

# Feature importance from logistic regression coefficients
coef_df = pd.DataFrame({"feature": FEATURES, "coefficient": logreg.coef_[0]}).sort_values("coefficient", key=abs, ascending=False)
print("Logistic regression coefficients (sign = direction, magnitude = strength):")
print(coef_df.to_string(index=False))

# Permutation importance as a cross-check
perm = permutation_importance(logreg, X_test, y_test, n_repeats=10, random_state=42)
perm_df = pd.DataFrame({"feature": FEATURES, "importance": perm.importances_mean}).sort_values("importance", ascending=False)
print("\nPermutation importance on held-out test set:")
print(perm_df.to_string(index=False))

# Error breakdown: false positives / false negatives at top-20
top20_idx = np.argsort(-logreg_scores)[:20]
top20_true = y_test[top20_idx]
print(f"\nOf model's top 20 flagged pages: {top20_true.sum()} actually declined, {20 - top20_true.sum()} did not (false positives).")


Logistic regression coefficients (sign = direction, magnitude = strength):
          feature  coefficient
       avg_clicks     0.031128
     avg_position    -0.025811
          avg_ctr    -0.005537
days_since_update    -0.004163
  avg_impressions    -0.000564
       word_count     0.000217
    search_volume     0.000005

Permutation importance on held-out test set:
          feature  importance
     avg_position    0.006867
          avg_ctr    0.000006
    search_volume   -0.000231
       avg_clicks   -0.004121
days_since_update   -0.016164
  avg_impressions   -0.022316
       word_count   -0.038201

Of model's top 20 flagged pages: 19 actually declined, 1 did not (false positives).


Interpretation: permutation importance shows only avg_position carries real predictive signal (importance 0.0069) every other feature (word_count, avg_impressions, days_since_update, avg_clicks) has negative importance, meaning the model performs better when those columns are shuffled into noise. This suggests the model is leaning almost entirely on current position rather than genuinely combining multiple signals.

The 95% precision at top-20 is high enough to be suspicious rather than simply good news. Since "declined" is defined as future avg_position being worse than current avg_position, pages that are already ranking poorly have structurally less room left to decline further the model may largely be learning this mechanical ceiling effect rather than an early-warning pattern a content team could act on before a page starts sliding. This is a case-in-point for the notebook 02 lesson: a suspiciously strong number is a prompt to check for a mechanical artifact, not a reason to trust the model more.

Next step (not done here): re-run with avg_position excluded from features, to see whether the remaining signals hold up at all without it carrying the whole result.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.